# Project 06 · Shop sales and ledger reconciliation

**Benson M. Merita · MeritaAnalytics** · Tools: Python (pandas), Matplotlib, Excel (output and a formula-based reconciliation workbook)

**Business question.** What are the agreed revenue, margin and growth figures, which products and stores carry the business, and do the numbers match the finance ledger?

**Data.** A 12-store retail chain in Kenya, 2025: two half-year sales extracts (8,035 raw lines, with errors), a product list (20 items), a store list (12 stores) and the finance team's monthly ledger (144 rows). Synthetic, generated with a fixed seed.

| Step | What it answers |
|---|---|
| 1 | Load the extracts and write a data quality report |
| 2 | Clean, with a row-count log |
| 3 | Merge safely with products and stores |
| 4 | Revenue, margin and monthly growth |
| 5 | Which products and stores carry the business? |
| 6 | Discounts and weekends |
| 7 | Reconcile to the finance ledger |
| 8 | Export to Excel |

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, GREEN, DARK, AMBER = "#1F5FD6", "#2BB38A", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

DATA = Path("../datasets/06-shop-sales")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
products = pd.read_csv(DATA / "reference/products.csv")
stores = pd.read_csv(DATA / "reference/stores.csv")
ledger = pd.read_csv(DATA / "raw/finance_ledger.csv")

## Step 1 · Load the extracts and write a data quality report

**What:** stack the two half-year files and report, per column, the data type, the number of blanks and the number of distinct values.
**Why:** look before you clean. This report shows the problems: blank stores, inconsistent payment labels and duplicated lines.

In [ ]:
files = sorted((DATA / "raw").glob("sales_*.csv"))
raw = pd.concat([pd.read_csv(p).assign(source_file=p.name) for p in files], ignore_index=True)
print("raw rows:", len(raw), "| exact duplicate rows:", int(raw.drop(columns="source_file").duplicated().sum()))
print("payment labels as typed:", sorted(raw["payment_method"].unique()))
pd.DataFrame({"dtype": raw.dtypes.astype(str), "blanks": raw.isna().sum(), "distinct": raw.nunique()})

## Step 2 · Clean, with a row-count log

**What:** remove duplicates, standardise IDs and payment labels, mark customers without a loyalty card, then set aside rows that cannot be used (bad date, missing store, zero or negative units), each with a reason.
**Why:** every row is accounted for, so anyone can audit the numbers: **raw = duplicates + rejected + clean**.

In [ ]:
log = [("raw rows read", len(raw))]
df = raw.drop_duplicates(subset=[c for c in raw.columns if c != "source_file"]).copy()
log.append(("after dropping duplicates", len(df)))

df["store_id"] = df["store_id"].str.strip().str.upper()
df["sku"] = df["sku"].str.strip().str.upper()
df["payment_method"] = df["payment_method"].str.strip().str.lower().map(
    {"mpesa": "M-Pesa", "m-pesa": "M-Pesa", "cash": "Cash", "card": "Card"})
df["customer_id"] = df["customer_id"].fillna("NO_CARD")
df["txn_date"] = pd.to_datetime(df["txn_date"], format="%Y-%m-%d", errors="coerce")

df["reject_reason"] = np.select([df["txn_date"].isna(), df["store_id"].isna(), df["units"] <= 0],
                                ["invalid date", "missing store", "non-positive units"], default="")
rejects = df[df["reject_reason"] != ""].copy()
df = df[df["reject_reason"] == ""].drop(columns="reject_reason").copy()
log.append(("after setting rejects aside", len(df)))
n_dups = len(raw) - log[1][1]

print("rejected by reason:", rejects["reject_reason"].value_counts().to_dict())
print("payment labels left unmapped:", int(df["payment_method"].isna().sum()))
pd.DataFrame(log, columns=["step", "rows"])

## Step 3 · Merge safely with products and stores

**What:** first a demonstration. If the product list accidentally contains a product twice, a plain merge silently adds rows. With `validate="many_to_one"`, pandas stops with an error instead. Then the real merge, followed by six checks.
**Why:** a silent row explosion inflates revenue without anyone noticing.

In [ ]:
bad_products = pd.concat([products, products.iloc[[0]]], ignore_index=True)    # one product listed twice
print("plain merge:", len(df), "->", len(df.merge(bad_products, on="sku", how="left")), "rows (silently inflated)")
try:
    df.merge(bad_products, on="sku", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("with validate: stopped ->", e)

In [ ]:
sales = (df.merge(products, on="sku", how="left", validate="many_to_one")
           .merge(stores, on="store_id", how="left", validate="many_to_one"))
sales["gross_ksh"] = sales["units"] * sales["list_price"]
sales["revenue_ksh"] = (sales["gross_ksh"] * (1 - sales["discount"])).round(2)
sales["cost_ksh"] = sales["units"] * sales["unit_cost"]
sales["margin_ksh"] = sales["revenue_ksh"] - sales["cost_ksh"]
sales["month"] = sales["txn_date"].dt.strftime("%Y-%m")

checks = {
    "transaction IDs unique": sales["txn_id"].is_unique,
    "all dates inside 2025": bool(sales["txn_date"].between("2025-01-01", "2025-12-31").all()),
    "all units positive": bool((sales["units"] > 0).all()),
    "every sale has a product and a store": bool(sales[["product_name", "store_name"]].notna().all().all()),
    "merge kept the row count": len(sales) == len(df),
    "raw = duplicates + rejected + clean": len(raw) == n_dups + len(rejects) + len(sales),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

## Step 4 · Revenue, margin and monthly growth

**What:** the headline KPIs, then revenue by month with month-on-month growth and a 3-month rolling average.

In [ ]:
revenue, margin, gross = sales["revenue_ksh"].sum(), sales["margin_ksh"].sum(), sales["gross_ksh"].sum()
print(f"lines {len(sales):,} | net revenue Ksh {revenue:,.2f} | gross margin Ksh {margin:,.0f} "
      f"({margin / revenue * 100:.1f}%) | discounts given Ksh {gross - revenue:,.0f}")

monthly = sales.groupby("month").agg(revenue_ksh=("revenue_ksh", "sum"), margin_ksh=("margin_ksh", "sum"))
monthly["mom_growth_pct"] = monthly["revenue_ksh"].pct_change() * 100
monthly["rolling3_ksh"] = monthly["revenue_ksh"].rolling(3).mean()
peak = monthly["revenue_ksh"].idxmax()
dec_ratio = monthly.loc["2025-12", "revenue_ksh"] / monthly.drop("2025-12")["revenue_ksh"].mean()
print(f"peak month {peak}; December is {dec_ratio:.2f}x the average of the other months")
monthly.round(1)

In [ ]:
fig, ax = plt.subplots()
ax.bar(monthly.index, monthly["revenue_ksh"] / 1e3, color=[AMBER if m == peak else GREEN for m in monthly.index])
ax.plot(monthly.index, monthly["rolling3_ksh"] / 1e3, color=DARK, marker="o", label="3-month rolling average")
ax.set_title("Monthly net revenue, 2025 (peak month in amber)"); ax.set_ylabel("Ksh thousand"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## Step 5 · Which products and stores carry the business?

**What:** a **Pareto (ABC) analysis** ranks products by revenue and finds how many make 80% of it. Then margin by category, and revenue per square metre of selling space by store (a fair way to compare stores of different sizes).

In [ ]:
prod = (sales.groupby(["product_name", "category"])
        .agg(revenue_ksh=("revenue_ksh", "sum"), margin_ksh=("margin_ksh", "sum"))
        .sort_values("revenue_ksh", ascending=False).reset_index())
prod["share_pct"] = prod["revenue_ksh"] / prod["revenue_ksh"].sum() * 100
prod["cum_share_pct"] = prod["share_pct"].cumsum()
prod["abc"] = np.select([prod["cum_share_pct"] <= 80, prod["cum_share_pct"] <= 95], ["A", "B"], "C")
n80 = int((prod["cum_share_pct"] <= 80).sum()) + 1
print(f"{n80} of {len(prod)} products make 80% of revenue; the top 3 make {prod['share_pct'].head(3).sum():.1f}%")
prod.head(8).round(1)

In [ ]:
cat = sales.groupby("category").agg(revenue_ksh=("revenue_ksh", "sum"), margin_ksh=("margin_ksh", "sum"))
cat["share_pct"] = cat["revenue_ksh"] / cat["revenue_ksh"].sum() * 100
cat["margin_pct"] = cat["margin_ksh"] / cat["revenue_ksh"] * 100
cat.sort_values("share_pct", ascending=False).round(1)

In [ ]:
store = sales.groupby(["store_name", "format", "area_sqm"]).agg(revenue_ksh=("revenue_ksh", "sum")).reset_index()
store["revenue_per_sqm"] = store["revenue_ksh"] / store["area_sqm"]
store = store.sort_values("revenue_per_sqm")
fig, ax = plt.subplots()
ax.barh(store["store_name"], store["revenue_per_sqm"],
        color=[AMBER if v == store["revenue_per_sqm"].min() else BLUE for v in store["revenue_per_sqm"]])
ax.set_title("Net revenue per square metre of selling area, 2025 (lowest in amber)"); ax.set_xlabel("Ksh per m²")
plt.tight_layout(); plt.show()

## Step 6 · Discounts and weekends

**What:** margin at each discount level, and average daily revenue on weekdays against weekends.
**Why:** deep discounts can sell more units while earning almost nothing.

In [ ]:
sales["discount_band"] = pd.cut(sales["discount"], [-0.01, 0, 0.05, 0.10, 0.20], labels=["0%", "5%", "10%", "20%"])
disc = sales.groupby("discount_band", observed=True).agg(lines=("txn_id", "count"), avg_units=("units", "mean"),
                                                         margin_ksh=("margin_ksh", "sum"), revenue_ksh=("revenue_ksh", "sum"))
disc["margin_pct"] = disc["margin_ksh"] / disc["revenue_ksh"] * 100
daily = sales.groupby("txn_date")["revenue_ksh"].sum()
dow = daily.groupby(daily.index.dayofweek >= 5).mean().rename({False: "weekday", True: "weekend"})
print(f"average daily revenue: weekday Ksh {dow['weekday']:,.0f} | weekend Ksh {dow['weekend']:,.0f} "
      f"({dow['weekend'] / dow['weekday']:.2f}x)")
disc[["lines", "avg_units", "margin_pct"]].round(2)

## Step 7 · Reconcile to the finance ledger

**What:** build a bridge from the ledger total to the cleaned pipeline total. The rejected rows explain part of the difference. Anything left is *unexplained*, so we compare ledger and pipeline for every store and month to find where it sits.
**Why:** finance will only sign off the analysis if it ties back to the books.

In [ ]:
ledger_total = round(ledger["revenue_ksh"].sum(), 2)
rj = rejects.merge(products[["sku", "list_price"]], on="sku", how="left", validate="many_to_one")
rejected_value = round((rj["units"].abs() * rj["list_price"] * (1 - rj["discount"])).sum(), 2)
pipeline_total = round(revenue, 2)
unexplained = round(ledger_total - rejected_value - pipeline_total, 2)
pd.DataFrame({"Ksh": [ledger_total, -rejected_value, -unexplained, pipeline_total]},
             index=["Finance ledger total", "Less rejected rows (documented)", "Less unexplained", "Cleaned pipeline revenue"])

In [ ]:
pipe = sales.groupby(["store_id", "month"], as_index=False)["revenue_ksh"].sum().rename(columns={"revenue_ksh": "pipeline_ksh"})
cmp_ = ledger.rename(columns={"revenue_ksh": "ledger_ksh"}).merge(pipe, on=["store_id", "month"], how="outer", validate="one_to_one")
cmp_["difference_ksh"] = (cmp_["ledger_ksh"] - cmp_["pipeline_ksh"]).round(2)
cmp_.reindex(cmp_["difference_ksh"].abs().sort_values(ascending=False).index).head(3)

One store-month stands out: **ST07, September 2025, a gap of about Ksh 85,000**. The small gaps elsewhere are the rejected rows. The same bridge is rebuilt with Excel formulas in `excel/06_shop_sales_reconciliation.xlsx`.

## Step 8 · Export to Excel

In [ ]:
xl = OUT / "06_shop_summary.xlsx"
with pd.ExcelWriter(xl) as writer:
    monthly.reset_index().to_excel(writer, sheet_name="Monthly", index=False)
    prod.to_excel(writer, sheet_name="Products", index=False)
    cmp_.to_excel(writer, sheet_name="Ledger vs pipeline", index=False)
    rejects.drop(columns="source_file").to_excel(writer, sheet_name="Rejected", index=False)
print({sheet: frame.shape for sheet, frame in pd.read_excel(xl, sheet_name=None).items()})

## Key findings and recommendations

- **Net revenue Ksh 4.87M at a 17.8% gross margin**; all six validation checks pass.
- **7 of 20 products make 80% of revenue**, and the top three make about 60%. *Secure supply and pricing for them.*
- **Dairy and Eggs** is the largest category but has the lowest margin.
- **Margin collapses at a 20% discount** (under 1%, against about 20% with no discount). *Cap deep discounts.*
- **December is about 1.4 times** other months and weekends about 1.2 times weekdays. *Plan stock and staff accordingly.*
- **Ask finance to explain the Ksh 85,000 posting in ST07, September 2025**, before the figures are signed off.

**Limitations.** Synthetic data, one year, fixed unit costs. The checks and reconciliation method are what transfer to real books.